In [32]:
import os
import numpy as np
import scipy.io as sio
from sklearn.utils import shuffle

## Official script

In [33]:
filenames = []
for filename in os.listdir("AFAVA-AD/"):
  filenames.append(filename)

In [34]:
filenames.sort()
# filenames

In [35]:
feature_path = 'Feature'
if not os.path.exists(feature_path):
    os.mkdir(feature_path)

#### Save feature

In [ ]:
subseq_length = 256
stride = 128  # Half of the subsequence length for half-overlapping
for i in range(len(filenames)):
    # print('Dataset/'+filename)
    path = "AFAVA-AD/" + filenames[i]
    mat = sio.loadmat(path)
    mat_np = mat['data']

    # Get epoch number for each subject
    epoch_num = len(mat_np[0,0][2][0])
    print("Epoch number: ",epoch_num)
    # Each epoch has shape (1280, 16)
    temp = np.zeros((epoch_num, 1280, 16))
    features = []
    # Store in temp
    for j in range(epoch_num):
        temp[j] = np.transpose(mat_np[0,0][2][0][j])

        # Calculate the number of subsequences that can be extracted
        num_subsequences = (temp[j].shape[0] - subseq_length) // stride + 1
        # Extract the subsequences
        subsequences = [temp[j][i * stride : i * stride + subseq_length, :] for i in range(num_subsequences)]
        feature = np.array(subsequences)
        features.append(feature)
    features = np.array(features).reshape((-1, subseq_length, 16))

    print(f"Filename: {filenames[i]}")
    print(f"Patient ID: {i+1}")
    print("Raw data:", temp.shape)
    print("Segmented data", features.shape)
    np.save(feature_path + "/feature_{:02d}.npy".format(i+1),features)
    print("Save feature_{:02d}.npy".format(i+1))
    print()

#### Save label

In [37]:
AD_positive = [1,3,6,8,9,11,12,13,15,17,19,21]

In [38]:
labels = np.zeros((23, 2))
len(labels)

23

In [39]:
label_path = 'Label'
if not os.path.exists(label_path):
    os.mkdir(label_path)

In [40]:
for i in range(len(labels)):
  # The first one is AD label (0 for healthy; 1 for AD patient)
  # The second one is the subject label (the order of subject, ranging from 1 to 23.
  labels[i][1] = i + 1
  if i+1 in AD_positive:
    labels[i][0] = 1
  else:
    labels[i][0] = 0

In [ ]:
np.save(label_path + "/label.npy",labels)
print("Save label")

## Test

In [42]:
test = np.load("Feature/feature_20.npy")

In [ ]:
test.shape

In [ ]:
test_label = np.load("Label/label.npy")
test_label